In [ ]:
import stim
import numpy as np
import matplotlib.pyplot as plt
from ldpc import bposd_decoder # note: old ldpc version was 0.1.45
# from ldpc.bplsd_decoder import BpLsdDecoder
# from ldpc.bposd_decoder import BpOsdDecoder
from decoder_helpers import get_shots_and_decode_info_from_circuit
%load_ext autoreload
%autoreload 2

# Simple surface code example

In [ ]:
num_shots = 500
p = 5e-3

circ = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=9,
    distance=3,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
    before_measure_flip_probability=p,
    before_round_data_depolarization=p)
detector_shots, observable_shots, det_mat, obs_mat, w, gauge_logical_index = (
    get_shots_and_decode_info_from_circuit(
        circ, num_shots=num_shots, decompose_errors=False))

In [ ]:
plt.plot(np.sum(det_mat, axis=0).T)
plt.xlabel('Detector label')
plt.ylabel('Number of errors triggering given detector')

In [ ]:
plt.plot(np.sum(det_mat, axis=1))
plt.xlabel('Error label')
plt.ylabel('Number of detectors triggering given error')

In [9]:
parity_check_matrix = det_mat.T.toarray().astype(np.uint8)
decoder_type = 'bposd'

if decoder_type == 'bposd':
    # decoder = BpOsdDecoder(
    #     pcm=parity_check_matrix,
    #     error_channel=list(w),
    #     bp_method='minimum_sum',
    #     ms_scaling_factor=0.9,
    #     osd_method="OSD_E",
    #     osd_order = 10
    # )
    
    # old BPOSD in LDPC v1
    decoder = bposd_decoder(
        parity_check_matrix = parity_check_matrix,
        channel_probs = w,
        bp_method='ps',
        ms_scaling_factor=0.9,
        osd_method="osd_e",
        osd_order = 10
    )

In [ ]:
print(f"Raw observable expectation: {np.mean(observable_shots)}")

In [11]:
results = np.zeros((num_shots, np.shape(obs_mat)[1]))
for i in range(num_shots):
    corr = decoder.decode(detector_shots[i, :].T)
    obs_corr = (corr @ obs_mat + observable_shots[i, :]) % 2
    results[i, :] = obs_corr

In [ ]:
print(f"Corrected observable expectation: {np.mean(results)}")

# HGP Stim circuit example

In [ ]:
from bposd.hgp import hgp
from .generate_hgp_circuit import generate_hgp_circuit

In [17]:
num_shots = 100
H = np.loadtxt('./codes_lib/hgp_34_n225_d4_h.txt')
eval_code = hgp(H, H)
num_cycles = 4

error_params = {"p_i": 1e-3, "p_state_p": 1e-3, "p_m": 1e-3,
                "p_CX": 1e-3, "p_idling_gate": 1e-3}
circuit_type = 'coloration'
circ = generate_hgp_circuit(eval_code, error_params,
                               num_cycles, circuit_type)
detector_shots, observable_shots, det_mat, obs_mat, w, gauge_logical_index = (
    get_shots_and_decode_info_from_circuit(circ, num_shots=num_shots))

In [ ]:
plt.plot(np.sum(det_mat, axis=0).T)
plt.xlabel('Detector label')
plt.ylabel('Number of errors triggering given detector')

In [ ]:
plt.plot(np.sum(det_mat, axis=1))
plt.xlabel('Error label')
plt.ylabel('Number of detectors triggered by given error')

In [20]:
parity_check_matrix = det_mat.T.toarray().astype(np.uint8)
decoder_type = 'bposd'

if decoder_type == 'bposd':
    # old BPOSD in LDPC v1
    decoder = bposd_decoder(
        parity_check_matrix = parity_check_matrix,
        channel_probs = w,
        bp_method='ps',
        ms_scaling_factor=0.9,
        osd_method="osd_e",
        osd_order = 10
    )

In [ ]:
print(f"Raw observable expectation: {np.mean(observable_shots)}")

In [ ]:
results = np.zeros((num_shots, np.shape(obs_mat)[1]))
for i in range(num_shots):
    corr = decoder.decode(detector_shots[i, :].T)
    obs_corr = (corr @ obs_mat + observable_shots[i, :]) % 2
    results[i, :] = obs_corr
    if i % 10 == 0:
        print(i)

In [ ]:
print(f"Corrected observable expectation: {np.mean(results)}")